# Clase #11

### Imports

Ejercicio #1:

Ejercicio #2: 

Ejercicio #3

Ejercico #4 

Usar VGG o YoloV8 (o disponible) para segmentar los elementow detectados en una imagen.

Verificar fotografias de mascotas, de personas, elementos de vestir...

El usuario carga la imagen y el algoritmo retorna la imagen segmentada

Ejercicio #5 

En la practica anterior, incluir un esquema de fine-tuning para mejorar la segmentacion. Que necesito?

Ejercicios Adicionales

1. Usar alguna arquitectura de CNN para hacer un tracker tipo esqueleto de la mano, usando la camara integrada (En entorno local)

2. Hacer uso de Yolo para detectar objetos en algun video que el usuario cargue al script

### Adicional 1. Esqueleto de la mano con la cámara integrada

Se utiliza **MediaPipe Hand Landmarker**, una solución basada en redes convolucionales: un detector de palmas y una red que estima 21 puntos de la mano. Se conectan la muñeca, los nudillos y las articulaciones para dibujar el esqueleto de hasta dos manos.

OpenCV captura la cámara y convierte BGR a RGB. Usamos el modo `VIDEO` para procesar secuencialmente los fotogramas de la cámara con marcas de tiempo crecientes; este modo aprovecha el seguimiento entre fotogramas. El efecto espejo facilita mover la mano frente a la pantalla.

Fuentes: [MediaPipe Hands](https://arxiv.org/abs/2006.10214) y [guía oficial de Python](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker/python).

In [1]:
def seguir_manos(indice_camara=0):
    """Muestra el esqueleto de las manos hasta presionar Q o Esc."""
    if not isinstance(indice_camara, int) or indice_camara < 0:
        raise ValueError("El índice de la cámara debe ser un entero no negativo.")

    import time
    from pathlib import Path
    from urllib.request import urlopen
    import cv2
    import mediapipe as mp

    # Guardar el modelo localmente y reutilizarlo en siguientes ejecuciones.
    modelo = Path("modelos/hand_landmarker.task")
    if not modelo.is_file():
        modelo.parent.mkdir(parents=True, exist_ok=True)
        url = ("https://storage.googleapis.com/mediapipe-models/"
               "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task")
        print("Descargando el modelo de manos...")
        with urlopen(url, timeout=60) as respuesta:
            contenido = respuesta.read()
        modelo.write_bytes(contenido)

    # Índices: muñeca 0; pulgar 1-4; índice 5-8; medio 9-12;
    # anular 13-16; meñique 17-20.
    conexiones = [
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 5), (5, 6), (6, 7), (7, 8),
        (5, 9), (9, 10), (10, 11), (11, 12),
        (9, 13), (13, 14), (14, 15), (15, 16),
        (13, 17), (0, 17), (17, 18), (18, 19), (19, 20),
    ]
    opciones = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(modelo.resolve())),
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
        num_hands=2,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    camara = cv2.VideoCapture(indice_camara)
    ventana = "Esqueleto de manos - Q o Esc para salir"
    ultimo_ms = -1
    try:
        if not camara.isOpened():
            raise RuntimeError("No se pudo abrir la cámara. Revisa permisos o prueba el índice 1.")
        with mp.tasks.vision.HandLandmarker.create_from_options(opciones) as detector:
            while True:
                disponible, fotograma = camara.read()
                if not disponible:
                    raise RuntimeError("No se pudo leer un fotograma de la cámara.")
                fotograma = cv2.flip(fotograma, 1)
                rgb = cv2.cvtColor(fotograma, cv2.COLOR_BGR2RGB)
                imagen = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                tiempo_ms = max(ultimo_ms + 1, time.monotonic_ns() // 1_000_000)
                ultimo_ms = tiempo_ms
                resultado = detector.detect_for_video(imagen, tiempo_ms)
                alto, ancho = fotograma.shape[:2]
                for mano in resultado.hand_landmarks:
                    puntos = [(int(p.x * ancho), int(p.y * alto)) for p in mano]
                    for inicio, fin in conexiones:
                        cv2.line(fotograma, puntos[inicio], puntos[fin], (0, 255, 0), 2)
                    for punto in puntos:
                        cv2.circle(fotograma, punto, 4, (0, 0, 255), -1)
                cv2.putText(fotograma, f"Manos: {len(resultado.hand_landmarks)}",
                            (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
                cv2.imshow(ventana, fotograma)
                if cv2.waitKey(1) & 0xFF in (ord("q"), ord("Q"), 27):
                    break
                if cv2.getWindowProperty(ventana, cv2.WND_PROP_VISIBLE) < 1:
                    break
    finally:
        camara.release()
        cv2.destroyAllWindows()

In [2]:
# La cámara integrada normalmente corresponde al índice 0.
seguir_manos(indice_camara=0)

Descargando el modelo de manos...


### Adicional 2. Detección de objetos en un video cargado por el usuario

Se utiliza **YOLOv8n**, una variante pequeña de YOLO preentrenada en COCO. Para cada fotograma se obtienen las cajas, clases y puntuaciones de confianza. `plot()` dibuja esas detecciones y OpenCV escribe el resultado en un archivo MP4.

La celda de ejecución abre un selector de archivos local mediante `tkinter`. También se puede pasar la ruta directamente. El umbral `confianza=0.35` descarta detecciones con puntuaciones menores. Se procesa un fotograma a la vez para evitar cargar todo el video en memoria; `device="cpu"` permite ejecutarlo sin GPU.

El resultado se guarda en `resultados/`, con el mismo tamaño y FPS del video (30 FPS si el archivo no informa una tasa válida). **No incluye audio**. Q o Esc guarda únicamente la parte procesada. La inferencia puede ser más lenta que la reproducción original. El modelo solo detecta las clases que aprendió durante su entrenamiento.

Fuente: [documentación de predicción de Ultralytics](https://docs.ultralytics.com/modes/predict/).

In [3]:
def detectar_video(ruta_video=None, confianza=0.35, salida=None, mostrar=True):
    """Selecciona un video, detecta objetos y devuelve la ruta del MP4 anotado."""
    import math
    from pathlib import Path
    from uuid import uuid4

    if not 0 < confianza <= 1:
        raise ValueError("La confianza debe estar entre 0 (excluido) y 1.")
    if ruta_video is None:
        import tkinter as tk
        from tkinter import filedialog

        raiz = tk.Tk()
        raiz.withdraw()
        try:
            ruta_video = filedialog.askopenfilename(
                parent=raiz, title="Selecciona el video para YOLO",
                filetypes=[("Videos", "*.mp4 *.avi *.mov *.mkv *.webm"),
                           ("Todos los archivos", "*.*")],
            )
        finally:
            raiz.destroy()
        if not ruta_video:
            print("Selección cancelada.")
            return None

    entrada = Path(ruta_video).expanduser().resolve()
    if not entrada.is_file():
        raise FileNotFoundError(f"No existe el video: {entrada}")
    if salida is None:
        salida = Path("resultados") / f"{entrada.stem}_yolo_{uuid4().hex[:8]}.mp4"
    salida = Path(salida).expanduser().resolve()
    if salida == entrada:
        raise ValueError("La salida debe ser diferente del video original.")
    if salida.suffix.lower() != ".mp4":
        raise ValueError("La salida debe tener extensión .mp4.")
    if salida.exists():
        raise FileExistsError(f"La salida ya existe; elige otro nombre: {salida}")

    import cv2
    from ultralytics import YOLO

    captura = cv2.VideoCapture(str(entrada))
    escritor = None
    procesados = 0
    ventana = "YOLO - Q o Esc para terminar"
    try:
        if not captura.isOpened():
            raise RuntimeError("No se pudo abrir el video; revisa su formato o códec.")
        disponible, fotograma = captura.read()
        if not disponible:
            raise RuntimeError("El video no contiene fotogramas legibles.")
        fps = captura.get(cv2.CAP_PROP_FPS)
        if not math.isfinite(fps) or fps <= 0:
            fps = 30.0
        alto, ancho = fotograma.shape[:2]
        # Los pesos se descargan automáticamente en la primera ejecución.
        modelo = YOLO("yolov8n.pt")
        salida.parent.mkdir(parents=True, exist_ok=True)
        escritor = cv2.VideoWriter(
            str(salida), cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto)
        )
        if not escritor.isOpened():
            raise RuntimeError("No se pudo crear el MP4; revisa el códec y la carpeta de salida.")
        while disponible:
            resultado = modelo.predict(
                source=fotograma, conf=confianza, device="cpu", verbose=False
            )[0]
            anotado = resultado.plot()
            escritor.write(anotado)
            procesados += 1
            if mostrar:
                cv2.imshow(ventana, anotado)
                if cv2.waitKey(1) & 0xFF in (ord("q"), ord("Q"), 27):
                    break
                if cv2.getWindowProperty(ventana, cv2.WND_PROP_VISIBLE) < 1:
                    break
            disponible, fotograma = captura.read()
    finally:
        captura.release()
        if escritor is not None:
            escritor.release()
        if mostrar:
            cv2.destroyAllWindows()

    print(f"Fotogramas procesados: {procesados}")
    print(f"Video con detecciones: {salida}")
    return salida

In [4]:
# Seleccionar un archivo del equipo; cancelar cierra el selector sin procesar.
video_resultante = detectar_video()

# Alternativa sin selector (útil si tkinter no está instalado):
# video_resultante = detectar_video(r"C:\Videos\ejemplo.mp4", mostrar=False)

Selección cancelada.


### Verificación opcional sin cámara ni modelos

Después de ejecutar las celdas que definen las dos funciones, esta celda comprueba los errores de entrada y la protección del video original. Estas pruebas no evalúan la precisión de los modelos ni sustituyen la ejecución con la cámara y un video real.

In [5]:
# Pruebas de validaci?n: no abren la c?mara ni descargan modelos.
import unittest
from pathlib import Path
from tempfile import TemporaryDirectory

class PruebasAdicionales(unittest.TestCase):
    def test_indice_camara_invalido(self):
        self.assertTrue(callable(globals().get("seguir_manos")), "Falta seguir_manos")
        with self.assertRaises(ValueError):
            seguir_manos(indice_camara=-1)

    def test_video_inexistente(self):
        self.assertTrue(callable(globals().get("detectar_video")), "Falta detectar_video")
        with TemporaryDirectory() as carpeta:
            with self.assertRaises(FileNotFoundError):
                detectar_video(Path(carpeta) / "no_existe.mp4")

    def test_confianza_invalida(self):
        self.assertTrue(callable(globals().get("detectar_video")), "Falta detectar_video")
        with self.assertRaises(ValueError):
            detectar_video("video.mp4", confianza=1.5)

    def test_proteger_video_original(self):
        self.assertTrue(callable(globals().get("detectar_video")), "Falta detectar_video")
        with TemporaryDirectory() as carpeta:
            original = Path(carpeta) / "video.mp4"
            original.write_bytes(b"contenido original")
            with self.assertRaises(ValueError):
                detectar_video(original, salida=original)
            self.assertEqual(original.read_bytes(), b"contenido original")

resultado_pruebas = unittest.TextTestRunner(verbosity=2).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(PruebasAdicionales)
)


test_confianza_invalida (__main__.PruebasAdicionales.test_confianza_invalida) ... ok
test_indice_camara_invalido (__main__.PruebasAdicionales.test_indice_camara_invalido) ... ok
test_proteger_video_original (__main__.PruebasAdicionales.test_proteger_video_original) ... ok
test_video_inexistente (__main__.PruebasAdicionales.test_video_inexistente) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.087s

OK
